[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-04-tmx-cleaning.ipynb)


# CS 479, Lecture 4: In-Class Activity
## Break the alignment, then fix it

**Ungraded. Work in pairs. About 15 minutes.**

This is a dry run of Assignment 4, on ten translation units instead of two hundred thousand.
The sample below is a TMX file with problems planted in it, the same problems the Church TM
data will hand you. Nothing is downloaded and no class data leaves this notebook.

**Before you start:** File > Save a copy in Drive, so you are editing your own copy.

By the end you will have found, by yourself, the failure that costs people a weekend:
the segment that quietly becomes two lines.


---
## Step 1. Install

About three seconds. Translate Toolkit is pure Python, so there is no large download here.


In [ ]:
!pip install -q translate-toolkit

## Step 2. The sample data

Ten translation units. Read the XML if you like, but do not fix anything by hand:
the whole point is to find these with code.


In [ ]:
SAMPLE_TMX = r"""<?xml version="1.0" encoding="utf-8"?>
<tmx version="1.4">
  <header creationtool="CS479demo" creationtoolversion="1.0" datatype="xml"
          segtype="sentence" adminlang="en-us" srclang="en-US" o-tmf="none"/>
  <body>
    <tu tuid="1">
      <tuv xml:lang="en-US"><seg>Come, follow me.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>Ven, sigueme.</seg></tuv></tu>
    <tu tuid="2">
      <tuv xml:lang="en-US"><seg>Faith is not a perfect knowledge&#xd;
of all things.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>La fe no es un conocimiento perfecto de todas las cosas.</seg></tuv></tu>
    <tu tuid="3">
      <tuv xml:lang="en-US"><seg>Click &lt;ph&gt;Next&lt;/ph&gt; to continue.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>Haga clic en &lt;ph&gt;Siguiente&lt;/ph&gt; para continuar.</seg></tuv></tu>
    <tu tuid="4">
      <tuv xml:lang="en-US"><seg>&quot;Peace&quot;&#160;be with you.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>&#171;Paz&#187; sea con vosotros.</seg></tuv></tu>
    <tu tuid="5">
      <tuv xml:lang="en-US"><seg>2 Nephi 31:20</seg></tuv>
      <tuv xml:lang="es-ES"><seg>2 Nefi 31:20</seg></tuv></tu>
    <tu tuid="6">
      <tuv xml:lang="en-US"><seg></seg></tuv>
      <tuv xml:lang="es-ES"><seg>Capitulo 3</seg></tuv></tu>
    <tu tuid="7">
      <tuv xml:lang="en-US"><seg>Study   the   word.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>Estudia la palabra.</seg></tuv></tu>
    <tu tuid="8">
      <tuv xml:lang="en-US"><seg>Charity never faileth.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>Charity never faileth.</seg></tuv></tu>
    <tu tuid="9">
      <tuv xml:lang="en-US"><seg>Behold, I say unto you&#xd;that ye must watch and pray always.</seg></tuv>
      <tuv xml:lang="es-ES"><seg>He aqui, os digo que debeis velar y orar siempre.</seg></tuv></tu>
    <tu tuid="10">
      <tuv xml:lang="en-US"><seg>[ Chapter 4</seg></tuv>
      <tuv xml:lang="es-ES"><seg>[ Capitulo 4</seg></tuv></tu>
  </body>
</tmx>
"""

with open("sample.tmx", "w", encoding="utf-8") as f:
    f.write(SAMPLE_TMX)

print("wrote sample.tmx:", len(SAMPLE_TMX), "characters")


## Step 3. Parse it and look

`tmxfile(f, "en-US", "es-ES")` reads the file; `unit_iter()` walks the translation units,
and each unit gives you `.source` and `.target`.

Notice the `!r` in the print below. That prints the **repr** of the string, so escape
characters show up instead of doing their job. Reading your data through `repr` is the
single most useful habit in this assignment.


In [ ]:
from translate.storage.tmx import tmxfile

with open("sample.tmx", "rb") as f:
    tmx = tmxfile(f, "en-US", "es-ES")

pairs = [(u.source, u.target) for u in tmx.unit_iter()]
print("translation units:", len(pairs), "\n")

for i, (en, es) in enumerate(pairs, 1):
    print(f"{i:>2}. {en!r}")
    print(f"    {es!r}")


## Step 4. Write the two files, the naive way

Assignment 4 asks for two sentence-aligned text files: one English line per one target line.
Write them the obvious way and count what comes out.

**Before you run this cell, predict the two numbers.**


In [ ]:
def write_pairs(pairs, en_path="english.txt", tx_path="spanish.txt"):
    with open(en_path, "w", encoding="utf-8") as fe, open(tx_path, "w", encoding="utf-8") as ft:
        for en, tx in pairs:
            fe.write(en + "\n")
            ft.write(tx + "\n")

def line_counts(en_path="english.txt", tx_path="spanish.txt"):
    en = open(en_path, encoding="utf-8").read().splitlines()
    tx = open(tx_path, encoding="utf-8").read().splitlines()
    print(f"{len(en)} lines in {en_path}")
    print(f"{len(tx)} lines in {tx_path}")
    print("aligned?", len(en) == len(tx))
    return en, tx

write_pairs(pairs)
en, tx = line_counts()


### What just happened

Ten units went in. The two files disagree, and neither one has ten lines.

Look back at the `repr` output from step 3. One segment carries `\r\n` and another carries a
bare `\r`, both of which were `&#xd;` in the XML. When you write that segment to a file,
it lands as two lines. Every pair after it is now off by one, and the two files have drifted
apart by a different amount each.

This is **cleaning step 1**, and it is why step 1 comes before everything else. Step 15,
misalignment, is just this problem noticed too late.


## Step 5. Your turn: repair it

Write the regular expression that finds those line breaks inside a segment.
Fill in `PATTERN`, run the cell, and get both counts to ten.


In [ ]:
import re

# TODO: a regular expression that matches the line breaks hiding inside a segment.
# Hint: you are looking for carriage returns and line feeds, one or more in a row.
PATTERN = None          # <-- replace None with r"..."
REPLACEMENT = " "

def repair(s):
    if PATTERN is None:
        return s
    return re.sub(PATTERN, REPLACEMENT, s).strip()

repaired = [(repair(en), repair(tx)) for en, tx in pairs]

if PATTERN is None:
    print("PATTERN is still None, so nothing was repaired. Fill it in above and re-run.")
else:
    write_pairs(repaired)
    line_counts()


## Step 6. What else is in there

Alignment was the emergency. It was not the only problem. This checker looks for a few of the
others and names the cleaning step each one belongs to.


In [ ]:
import re

def inspect(pairs):
    findings = []
    for i, (en, tx) in enumerate(pairs, 1):
        if not en.strip() or not tx.strip():
            findings.append((i, "step 2", "one side is empty"))
        if "\xa0" in en or "\xa0" in tx:
            findings.append((i, "step 4", "non-breaking space (this was &nbsp; in the file)"))
        if re.search(r"   *", en) or re.search(r"   *", tx):
            findings.append((i, "step 4", "runs of spaces"))
        if re.search(r"<[^>]+>", en) or re.search(r"<[^>]+>", tx):
            findings.append((i, "step 6", "an inline tag survived"))
        if en.strip() and en.strip() == tx.strip():
            findings.append((i, "step 10", "source equals target"))
        for opener, closer in [("(", ")"), ("[", "]"), ("{", "}")]:
            if en.count(opener) != en.count(closer) or tx.count(opener) != tx.count(closer):
                findings.append((i, "step 11", f"unbalanced {opener}{closer}"))
    return findings

for i, step, what in inspect(repaired):
    print(f"unit {i:>2}  ({step})  {what}")


## Step 7. Your turn again: clean two of them

Pick any two findings and write a small function for each. Keep them separate:
your Assignment 5 pipeline will be a stack of exactly these.


In [ ]:
# TODO: pick two of the problems above and write a cleaner for each.
# Keep each one a separate small function, the way your pipeline will want them.

def clean_one(s):
    # e.g. normalize whitespace, or strip inline tags
    return s

def clean_two(s):
    return s

cleaned = [(clean_two(clean_one(en)), clean_two(clean_one(tx))) for en, tx in repaired]

remaining = inspect(cleaned)
print(f"{len(inspect(repaired))} findings before, {len(remaining)} after\n")
for i, step, what in remaining:
    print(f"unit {i:>2}  ({step})  {what}")


## Step 8. Report back

We will take answers as a class.

1. What was your `PATTERN`, and did it also catch the bare `\r` in unit 9?
2. Which finding surprised you, and which cleaning step does it belong to?
3. Unit 8 is the same text on both sides. Why is that worse than useless for training?
4. Unit 5 is a scripture reference, the same on both sides apart from one word. Keep it or drop it?
   Defend your answer, because this is a judgment call your data will ask you to make hundreds of times.


---
## Appendix A. No package at all

TMX is XML, and `xml.etree` is in the standard library. If Translate Toolkit fights your
environment, this is a complete parser in eight lines, and writing your own is explicitly
allowed on the assignment.


In [ ]:
import xml.etree.ElementTree as ET

root = ET.parse("sample.tmx").getroot()
own_pairs = []
for tu in root.iter("tu"):
    segs = [tuv.find("seg").text or "" for tuv in tu.findall("tuv")]
    if len(segs) == 2:
        own_pairs.append((segs[0], segs[1]))

print(len(own_pairs), "pairs, no third-party package involved")
print(own_pairs[1])


## Appendix B. A note on python-tmx

The `python-tmx` package on PyPI installs as `PythonTmx`, and its current API is
`from_element(root)` over an ElementTree element, not a `from_tmx` method. Its parser is also
strict: a TMX whose header is missing `creationtoolversion` raises `KeyError` rather than
warning you. Usable, but Translate Toolkit or plain `xml.etree` will cost you less time today.


---
### A reminder on AI use

Per Lecture 1 and the assignment: you may ask AI what a library function does. Do not have it
write your algorithm or your code, and be ready to explain every line you submit. A regular
expression a model wrote and you cannot read is a bug you will not be able to find.

BYU CS AI policy: https://cs.byu.edu/department/ai-policy
